# LD-3 — FUMÉE Kaggle CPU : une page leurre, référence B3 factice

In [ ]:

import time
T_START = time.monotonic()          # pris au tout début de la cellule
import datetime, faulthandler, hashlib, json, os, shlex, shutil, subprocess, sys, zipfile
from pathlib import Path
faulthandler.enable()

def _utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')

print('ld3 start', _utc(), flush=True)
OUT = Path('/kaggle/working')
KIT_CANDIDATES = ['/kaggle/input/babaorum-ld3-kit', '/kaggle/input/datasets/mabaku/babaorum-ld3-kit']
PAGES_CANDIDATES = ['/kaggle/input/babaorum-e2e3-run3-fp-classes/e2e3/mf', '/kaggle/input/notebooks/mabaku/babaorum-e2e3-run3-fp-classes/e2e3/mf']
CKPT_CANDIDATES = ['/kaggle/input/babaorum-e2e5-lambda/logs/checkpoint_last.pt', '/kaggle/input/notebooks/mabaku/babaorum-e2e5-lambda/logs/checkpoint_last.pt']
LD2_CANDIDATES = ['/kaggle/input/babaorum-ld2-run/ld2', '/kaggle/input/notebooks/mabaku/babaorum-ld2-run/ld2']
E2E4_CANDIDATES = ['/kaggle/input/babaorum-e2e4-extraction/e2e4_extraction_report.json', '/kaggle/input/notebooks/mabaku/babaorum-e2e4-extraction/e2e4_extraction_report.json']
BUNDLE_CANDIDATES = ['/kaggle/input/babaorum-stage-s-bundle-v5/stage_s_bundle_v5.zip.bin', '/kaggle/input/datasets/mabaku/babaorum-stage-s-bundle-v5/stage_s_bundle_v5.zip.bin']
E2E5_SHA = 'ef5e826803c347f73e49c83c0f21f61399f9ea06ee985905bd17644dff3844fa'
LD2_SHA = 'e5afd1432e10aa8e06191678844ae350348f7586f505d1a766aba3a5a44b42c3'
LD2_FILE = 'ld2_main_step003271.pt'
E2E4_SHA = '714bc571db6a4e394027ec4e98cd4c997bde5ff54c735fbc8f32d9cdc4218654'
BUNDLE_SHA = 'ca41c09272248c5096f2547fe738eee622db5af50414182537ac8e94e43367f5'
COMMIT = '23875ab05425ce2aa07742bf314d02436c2f5642'
REQUIRED_PIP = ['Pillow>=12.3,<13', 'fonttools>=4.60,<5', 'jsonschema>=4.25,<5', 'lxml>=6,<7', 'numpy>=2.3,<3', 'shapely>=2.1,<3']
TIMEOUTS = {'pip': 1500, 'unzip': 900, 'clone': 600, 'controls': 600, 'D': 900, 'CAL': 1800, 'generate': 1500}
LIMITS = {'deadline_s': 9000, 'stop_margin_s': 120, 'eval_margin_s': 300, 'estimate_factor': 1.5, 'generation_estimate_s': 600, 'heartbeat_s': 60}
RUN_FILES = ['scripts/eval_ld3.py', 'scripts/data3_generate.py', 'config/ld3.json', 'tests/test_ld3.py']
def resolve_one(candidates, marker):
    """Exactement un candidat dont <candidat>/<marqueur> existe. Aucun listage."""
    hits = [Path(c) for c in candidates if (Path(c) / marker).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"montage : {len(hits)} candidat(s) pour {marker}")
    return hits[0]

def resolve_file(candidates):
    """Exactement un candidat qui est un fichier (§12 (15))."""
    hits = [Path(c) for c in candidates if Path(c).is_file()]
    if len(hits) != 1:
        raise RuntimeError(f"fichier : {len(hits)} candidat(s)")
    return hits[0]


def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

KIT = resolve_one(KIT_CANDIDATES, "config/ld3.json")
PAGES = resolve_one(PAGES_CANDIDATES, "pages/mf_0000.json")
CKPT = resolve_file(CKPT_CANDIDATES)
LD2DIR = resolve_one(LD2_CANDIDATES, "ld2_manifest.json")
E2E4 = resolve_file(E2E4_CANDIDATES)
BUNDLE_ZIP = resolve_file(BUNDLE_CANDIDATES)
assert sha256(CKPT) == E2E5_SHA, "checkpoint E2E-5 : sha différent"
assert sha256(LD2DIR / LD2_FILE) == LD2_SHA, "checkpoint LD-2 : sha différent"
assert sha256(E2E4) == E2E4_SHA, "rapport e2e4 : sha différent"
assert sha256(BUNDLE_ZIP) == BUNDLE_SHA, "bundle v5 : sha différent"
sys.path.insert(0, str(KIT / "src"))
sys.path.insert(0, str(KIT / "scripts"))
import importlib.util as _u
_spec = _u.spec_from_file_location("data2_common", KIT / "scripts" / "data2_common.py")
C = _u.module_from_spec(_spec); _spec.loader.exec_module(C)
import ld2_core as C2

BUNDLE_DIR = Path("/tmp/ld3_bundle")
SRC = Path("/tmp/ld3_heritage")
LIBS = Path("/tmp/ld3_genlib")
WORK = Path("/tmp/ld3_work")

def _argv(phase, extra=()):
    return [sys.executable, str(KIT / "scripts" / "eval_ld3.py"), "--phase", phase, "--root", str(PAGES),
            "--out", str(RUN), "--ld2-dir", str(LD2DIR), "--e2e5-checkpoint", str(CKPT), "--e2e4-report", str(E2E4),
            "--gen-src", str(SRC), "--work", str(WORK), "--extra-libs", str(LIBS), "--cal-out", str(CAL_OUT),
            "--device", "cpu", "--cache-dir", "/tmp/ld3_hf", *extra]

def d_mille_check():
    """Les 51 fichiers suivis du générateur cloné égalent bundle_manifest.json (comme DATA-2)."""
    manifest = json.loads((BUNDLE_DIR / "bundle_manifest.json").read_text())
    for rel, want in manifest["mille_feuilles_sha256"].items():
        assert sha256(SRC / rel[len("heritage-synth/"):]) == want, ("fichier du générateur différent", rel)
    print("générateur : fichiers contrôlés", len(manifest["mille_feuilles_sha256"]), flush=True)

GEN_CMDS = {
    "pip": [sys.executable, "-m", "pip", "install", "--quiet", "--timeout", "30", "--retries", "1", "--target", str(LIBS),
            *REQUIRED_PIP],
    "unzip": [sys.executable, "-c", "import sys, zipfile; zipfile.ZipFile(sys.argv[1]).extractall(sys.argv[2])",
              str(BUNDLE_ZIP), str(BUNDLE_DIR)],
    "clone": ["git", "clone", "-q", str(BUNDLE_DIR / "heritage-synth.gitbundle"), str(SRC)],
    "checkout": ["git", "-C", str(SRC), "checkout", "-q", COMMIT],
}
GEN_CAPS = {"pip": TIMEOUTS["pip"], "unzip": TIMEOUTS["unzip"], "clone": TIMEOUTS["clone"], "checkout": TIMEOUTS["clone"]}
# Préparation à plafonds FIXES : utilisée telle quelle par la FUMÉE (CPU, aucune échéance globale de cellule à tenir). Le run
# complet reconstruit ces quatre étapes avec step_argv (plafond = restant - marge d'arrêt).
GEN_PREP = [*({"name": n, "argv": GEN_CMDS[n], "timeout_s": GEN_CAPS[n]} for n in ("pip", "unzip", "clone", "checkout")),
            {"name": "mille_check", "fn": d_mille_check}]

SMOKE = OUT / "ld3_smoke"
RUN = SMOKE / "ld3"
CAL_OUT = SMOKE / "ld3_cal"
SMOKE.mkdir(parents=True, exist_ok=True)
# FUMÉE : plafonds FIXES de la config (CPU, aucune échéance globale de cellule) ; seul le run complet borne chaque étape par le restant.
HEAVY_ENV = dict(os.environ, PYTHONPATH=str(KIT / "src") + os.pathsep + str(KIT / "scripts"))
HEAVY_ENV["HF_HUB_OFFLINE"] = "1"
OFFLINE = lambda: dict(os.environ, HF_HUB_OFFLINE="1")

def _smoke_extra(phase):
    return {"CAL": ["--pages", "mf_0000", "--smoke", "--eval-pages", "decoy_0000"],
            "EVAL": ["--smoke", "--decoy-reference"]}.get(phase, [])

STEPS = [
    *GEN_PREP,
    {"name": "D", "argv": _argv("D"), "timeout_s": TIMEOUTS["D"]},
    # tests/test_ld3.py complet, sans -k ; rc écrit dans un fichier, sans bloquer la suite
    {"name": "pytest", "argv": ["sh", "-c", "cd \"" + str(KIT) + "\" && " + sys.executable + " -m pytest -q -rs tests/test_ld3.py > \""
                                + str(SMOKE) + "/ld3_pytest.txt\" 2>&1; echo $? > \"" + str(SMOKE) + "/ld3_pytest_rc.txt\"; exit 0"],
     "env": HEAVY_ENV, "timeout_s": TIMEOUTS["CAL"]},
    {"name": "controls", "argv": _argv("controls"), "timeout_s": TIMEOUTS["controls"], "env": OFFLINE},
    {"name": "CAL", "argv": _argv("CAL", _smoke_extra("CAL")), "timeout_s": 5400, "env": OFFLINE},
    {"name": "EVAL", "argv": _argv("EVAL", _smoke_extra("EVAL")), "timeout_s": 10800, "env": OFFLINE},
]
SUMMARY = {"kind": "SMOKE_KAGGLE_CPU_PAGE_LEURRE", "e2e5_sha256": E2E5_SHA, "ld2_sha256": LD2_SHA}
EXIT = None
try:
    EXIT = C.execute_pipeline(SMOKE, STEPS, heartbeat_interval_s=LIMITS["heartbeat_s"])
finally:
    SUMMARY["pipeline"] = {k: EXIT[k] for k in ("status", "step", "exit_code", "durations_s")} if EXIT else None
    for name in ("ld3_pytest.txt", "ld3_pytest_rc.txt"):
        f = SMOKE / name
        if f.is_file():
            SUMMARY[name] = f.read_text()[-6000:]
    for name in ("ld3_report.json", "ld3_edge_table.json", "data3_generation_manifest.json", "data3_overlap_report.json",
                 "ld3_eval_started.json"):
        f = RUN / name
        if f.is_file():
            SUMMARY[name] = json.loads(f.read_text())
    cal = CAL_OUT / "ld3_cal.json"
    if cal.is_file():
        SUMMARY["ld3_cal.json"] = json.loads(cal.read_text())
    (OUT / "ld3_smoke_summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=str))
    for d in (BUNDLE_DIR, SRC, LIBS, WORK):
        shutil.rmtree(d, ignore_errors=True)
print("SMOKE", SUMMARY.get("pipeline"), flush=True)
